# 02 — BPE Tokenization

## Scientific Abstract GPT

This notebook trains a Byte-Level Byte Pair Encoding tokenizer from scratch
using the structured scientific text created in `01_Data_Preprocessing.ipynb`.

### Structural and control tokens

- `<PAD>` — padding token
- `<UNK>` — unknown token
- `<TITLE>` — beginning of the paper title
- `<SUBJECT>` — beginning of the research subject
- `<ABSTRACT>` — beginning of the abstract
- `<END>` — end of one scientific-paper record

### Processing steps

1. Load the structured dataset.
2. Train a Byte-Level BPE tokenizer using only the training split.
3. Save and reload the tokenizer.
4. Verify structural tokens.
5. Tokenize the train, validation, and test splits.
6. Save the tokenized Hugging Face dataset.
7. Create compact token-stream files for GPT training.
8. Save tokenizer configuration and tokenization statistics.


In [1]:
!pip install -q datasets tokenizers

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
import json
import shutil
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch

from datasets import (
    load_from_disk,
    DatasetDict
)

from tokenizers import (
    Tokenizer,
    ByteLevelBPETokenizer
)

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Random seed:", SEED)

Random seed: 42


In [5]:
PROJECT_PATH = (
    "/content/drive/MyDrive/"
    "Scientific-Abstract-GPT"
)

DATA_FOLDER = os.path.join(
    PROJECT_PATH,
    "data"
)

PROCESSED_DATASET_PATH = os.path.join(
    DATA_FOLDER,
    "processed_structured_dataset"
)

TOKENIZER_DATA_PATH = os.path.join(
    DATA_FOLDER,
    "tokenizer_data"
)

TRAIN_TEXT_PATH = os.path.join(
    TOKENIZER_DATA_PATH,
    "train_text.txt"
)

VALIDATION_TEXT_PATH = os.path.join(
    TOKENIZER_DATA_PATH,
    "validation_text.txt"
)

TEST_TEXT_PATH = os.path.join(
    TOKENIZER_DATA_PATH,
    "test_text.txt"
)

BPE_TOKENIZER_PATH = os.path.join(
    DATA_FOLDER,
    "bpe_tokenizer"
)

TOKENIZER_JSON_PATH = os.path.join(
    BPE_TOKENIZER_PATH,
    "tokenizer.json"
)

TOKENIZER_CONFIG_PATH = os.path.join(
    BPE_TOKENIZER_PATH,
    "tokenizer_config.json"
)

TOKENIZED_DATASET_PATH = os.path.join(
    DATA_FOLDER,
    "tokenized_bpe_dataset"
)

TOKEN_STREAM_PATH = os.path.join(
    DATA_FOLDER,
    "tokenized_bpe_streams"
)

TOKENIZATION_SUMMARY_PATH = os.path.join(
    DATA_FOLDER,
    "tokenization_summary.json"
)

os.makedirs(
    DATA_FOLDER,
    exist_ok=True
)

os.makedirs(
    BPE_TOKENIZER_PATH,
    exist_ok=True
)

os.makedirs(
    TOKEN_STREAM_PATH,
    exist_ok=True
)

print("Processed dataset:", PROCESSED_DATASET_PATH)
print("Training text:", TRAIN_TEXT_PATH)
print("BPE tokenizer folder:", BPE_TOKENIZER_PATH)
print("Tokenized dataset:", TOKENIZED_DATASET_PATH)
print("Token streams:", TOKEN_STREAM_PATH)

Processed dataset: /content/drive/MyDrive/Scientific-Abstract-GPT/data/processed_structured_dataset
Training text: /content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenizer_data/train_text.txt
BPE tokenizer folder: /content/drive/MyDrive/Scientific-Abstract-GPT/data/bpe_tokenizer
Tokenized dataset: /content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenized_bpe_dataset
Token streams: /content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenized_bpe_streams


Verify Notebook 1 Outputs

In [6]:
required_inputs = {
    "processed_dataset": (
        PROCESSED_DATASET_PATH
    ),
    "training_text": (
        TRAIN_TEXT_PATH
    ),
    "validation_text": (
        VALIDATION_TEXT_PATH
    ),
    "test_text": (
        TEST_TEXT_PATH
    )
}

missing_inputs = []

for input_name, input_path in (
    required_inputs.items()
):

    exists = os.path.exists(
        input_path
    )

    print(
        f"{input_name}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    if not exists:
        missing_inputs.append(
            input_path
        )

if missing_inputs:

    raise FileNotFoundError(
        "The following Notebook 1 outputs "
        f"were not found:\n{missing_inputs}"
    )

print(
    "\nAll required preprocessing "
    "outputs are available."
)

processed_dataset: FOUND
training_text: FOUND
validation_text: FOUND
test_text: FOUND

All required preprocessing outputs are available.


Load the Structured Dataset

In [7]:
structured_dataset = load_from_disk(
    PROCESSED_DATASET_PATH
)

print(structured_dataset)

DatasetDict({
    train: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count'],
        num_rows: 178549
    })
    validation: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count'],
        num_rows: 9919
    })
    test: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count'],
        num_rows: 9920
    })
})


In [8]:
REQUIRED_SPLITS = {
    "train",
    "validation",
    "test"
}

missing_splits = (
    REQUIRED_SPLITS -
    set(structured_dataset.keys())
)

if missing_splits:

    raise ValueError(
        "Missing dataset splits: "
        f"{sorted(missing_splits)}"
    )

for split_name in [
    "train",
    "validation",
    "test"
]:

    split_columns = (
        structured_dataset[
            split_name
        ].column_names
    )

    if "text" not in split_columns:

        raise ValueError(
            f"The '{split_name}' split "
            "does not contain a text column."
        )

    print(
        f"{split_name.capitalize()}: "
        f"{len(structured_dataset[split_name]):,} records"
    )

Train: 178,549 records
Validation: 9,919 records
Test: 9,920 records


In [9]:
sample_text = (
    structured_dataset[
        "train"
    ][0]["text"]
)

print(sample_text[:1500])

<TITLE> Contrastive Language Adaptation for Cross-Lingual Stance Detection <SUBJECT> Computation and Language <ABSTRACT> We study cross-lingual stance detection, which aims to leverage labeled data in one language to identify the relative perspective (or stance) of a given document with respect to a claim in a different target language. In particular, we introduce a novel contrastive language adaptation approach applied to memory networks, which ensures accurate alignment of stances in the source and target languages, and can effectively deal with the challenge of limited labeled data in the target language. The evaluation results on public benchmark datasets and comparison against current state-of-the-art approaches demonstrate the effectiveness of our approach. <END>


Define Tokenizer Configuration

In [10]:
VOCAB_SIZE = 8000
MIN_FREQUENCY = 2

SPECIAL_TOKENS = [
    "<PAD>",
    "<UNK>",
    "<TITLE>",
    "<SUBJECT>",
    "<ABSTRACT>",
    "<END>"
]

print("Target vocabulary size:", VOCAB_SIZE)
print("Minimum frequency:", MIN_FREQUENCY)

print("\nSpecial tokens:")

for token in SPECIAL_TOKENS:
    print("-", token)

Target vocabulary size: 8000
Minimum frequency: 2

Special tokens:
- <PAD>
- <UNK>
- <TITLE>
- <SUBJECT>
- <ABSTRACT>
- <END>


Inspect Training Text File

In [11]:
with open(
    TRAIN_TEXT_PATH,
    "r",
    encoding="utf-8"
) as file:

    for example_number in range(2):

        line = file.readline()

        print(
            f"Training line "
            f"{example_number + 1}:"
        )

        print(line[:1200])

        print("=" * 100)


Training line 1:
<TITLE> Contrastive Language Adaptation for Cross-Lingual Stance Detection <SUBJECT> Computation and Language <ABSTRACT> We study cross-lingual stance detection, which aims to leverage labeled data in one language to identify the relative perspective (or stance) of a given document with respect to a claim in a different target language. In particular, we introduce a novel contrastive language adaptation approach applied to memory networks, which ensures accurate alignment of stances in the source and target languages, and can effectively deal with the challenge of limited labeled data in the target language. The evaluation results on public benchmark datasets and comparison against current state-of-the-art approaches demonstrate the effectiveness of our approach. <END>

Training line 2:
<TITLE> Do Neural Networks Need Gradient Descent to Generalize? A Theoretical Study <SUBJECT> Machine Learning <ABSTRACT> Conventional wisdom attributes the mysterious generalization ab

In [12]:
def count_non_empty_lines(
    file_path
):

    count = 0

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as file:

        for line in file:

            if line.strip():
                count += 1

    return count

In [13]:
training_text_lines = (
    count_non_empty_lines(
        TRAIN_TEXT_PATH
    )
)

validation_text_lines = (
    count_non_empty_lines(
        VALIDATION_TEXT_PATH
    )
)

test_text_lines = (
    count_non_empty_lines(
        TEST_TEXT_PATH
    )
)

print(
    "Training text lines:",
    f"{training_text_lines:,}"
)

print(
    "Validation text lines:",
    f"{validation_text_lines:,}"
)

print(
    "Test text lines:",
    f"{test_text_lines:,}"
)

Training text lines: 178,549
Validation text lines: 9,919
Test text lines: 9,920


Remove Existing Tokenizer Files

In [14]:
if os.path.exists(
    BPE_TOKENIZER_PATH
):

    print(
        "Removing existing tokenizer folder:"
    )

    print(BPE_TOKENIZER_PATH)

    shutil.rmtree(
        BPE_TOKENIZER_PATH
    )

os.makedirs(
    BPE_TOKENIZER_PATH,
    exist_ok=True
)

print(
    "Tokenizer output folder is ready."
)

Removing existing tokenizer folder:
/content/drive/MyDrive/Scientific-Abstract-GPT/data/bpe_tokenizer
Tokenizer output folder is ready.


Create Byte-Level BPE Tokenizer

In [15]:
bpe_tokenizer = (
    ByteLevelBPETokenizer(
        add_prefix_space=False
    )
)

print(
    "Byte-Level BPE tokenizer created."
)

Byte-Level BPE tokenizer created.


Train the BPE Tokenizer

The tokenizer must be trained using the training split only.

This prevents information from the validation or test splits from being used during tokenizer training.

In [16]:
bpe_tokenizer.train(
    files=[
        TRAIN_TEXT_PATH
    ],
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=SPECIAL_TOKENS,
    show_progress=True
)

print(
    "BPE tokenizer training completed."
)

BPE tokenizer training completed.


Save Vocabulary and Merge Files

In [17]:
saved_tokenizer_files = (
    bpe_tokenizer.save_model(
        BPE_TOKENIZER_PATH
    )
)

print("Saved tokenizer files:")

for file_path in saved_tokenizer_files:
    print("-", file_path)

Saved tokenizer files:
- /content/drive/MyDrive/Scientific-Abstract-GPT/data/bpe_tokenizer/vocab.json
- /content/drive/MyDrive/Scientific-Abstract-GPT/data/bpe_tokenizer/merges.txt


Save Complete Tokenizer JSON

In [20]:
bpe_tokenizer.save(
    TOKENIZER_JSON_PATH
)

print(
    "Complete tokenizer saved:"
)

print(
    TOKENIZER_JSON_PATH
)

Complete tokenizer saved:
/content/drive/MyDrive/Scientific-Abstract-GPT/data/bpe_tokenizer/tokenizer.json


Reload the Saved Tokenizer

In [21]:
tokenizer = Tokenizer.from_file(
    TOKENIZER_JSON_PATH
)

actual_vocab_size = (
    tokenizer.get_vocab_size()
)

print(
    "Reloaded tokenizer successfully."
)

print(
    "Actual vocabulary size:",
    actual_vocab_size
)

Reloaded tokenizer successfully.
Actual vocabulary size: 8000


Get Special Token IDs

In [22]:
special_token_ids = {
    token: tokenizer.token_to_id(
        token
    )
    for token in SPECIAL_TOKENS
}

for token, token_id in (
    special_token_ids.items()
):

    print(
        f"{token}: {token_id}"
    )

<PAD>: 0
<UNK>: 1
<TITLE>: 2
<SUBJECT>: 3
<ABSTRACT>: 4
<END>: 5


Validate Special Token IDs

In [23]:
invalid_special_tokens = [
    token
    for token, token_id
    in special_token_ids.items()
    if token_id is None
]

if invalid_special_tokens:

    raise ValueError(
        "The following special tokens "
        "are missing from the tokenizer: "
        f"{invalid_special_tokens}"
    )

if len(
    set(
        special_token_ids.values()
    )
) != len(SPECIAL_TOKENS):

    raise ValueError(
        "Special tokens do not have "
        "unique token IDs."
    )

print(
    "All special tokens have unique IDs."
)

All special tokens have unique IDs.


Test Tokenization on a Scientific Example

In [24]:
test_scientific_text = (
    "<TITLE> Deep Learning for "
    "Medical Image Classification "
    "<SUBJECT> Machine Learning "
    "<ABSTRACT> This paper presents "
    "a transformer-based approach for "
    "medical image classification. "
    "<END>"
)

test_encoding = tokenizer.encode(
    test_scientific_text
)

print("Original text:")
print(test_scientific_text)

print("\nToken IDs:")
print(test_encoding.ids)

print("\nTokens:")
print(test_encoding.tokens)

print("\nNumber of tokens:")
print(len(test_encoding.ids))

Original text:
<TITLE> Deep Learning for Medical Image Classification <SUBJECT> Machine Learning <ABSTRACT> This paper presents a transformer-based approach for medical image classification. <END>

Token IDs:
[2, 1290, 468, 324, 5478, 3843, 2706, 226, 3, 536, 468, 226, 4, 624, 625, 2122, 264, 2599, 18, 659, 547, 324, 2154, 1690, 946, 19, 226, 5]

Tokens:
['<TITLE>', 'ĠDeep', 'ĠLearning', 'Ġfor', 'ĠMedical', 'ĠImage', 'ĠClassification', 'Ġ', '<SUBJECT>', 'ĠMachine', 'ĠLearning', 'Ġ', '<ABSTRACT>', 'ĠThis', 'Ġpaper', 'Ġpresents', 'Ġa', 'Ġtransformer', '-', 'based', 'Ġapproach', 'Ġfor', 'Ġmedical', 'Ġimage', 'Ġclassification', '.', 'Ġ', '<END>']

Number of tokens:
28


Decode the Test Example

In [25]:
decoded_test_text = tokenizer.decode(
    test_encoding.ids,
    skip_special_tokens=False
)

print("Original:")
print(test_scientific_text)

print("\nDecoded:")
print(decoded_test_text)

Original:
<TITLE> Deep Learning for Medical Image Classification <SUBJECT> Machine Learning <ABSTRACT> This paper presents a transformer-based approach for medical image classification. <END>

Decoded:
<TITLE> Deep Learning for Medical Image Classification <SUBJECT> Machine Learning <ABSTRACT> This paper presents a transformer-based approach for medical image classification. <END>


Verify Structural Tokens Are Atomic

In [26]:
for token in [
    "<TITLE>",
    "<SUBJECT>",
    "<ABSTRACT>",
    "<END>"
]:

    encoded_token = tokenizer.encode(
        token
    )

    print(
        token,
        "->",
        encoded_token.ids,
        "->",
        encoded_token.tokens
    )

    if len(encoded_token.ids) != 1:

        raise ValueError(
            f"{token} was split into "
            "multiple tokens."
        )

print(
    "\nAll structural tokens are atomic."
)

<TITLE> -> [2] -> ['<TITLE>']
<SUBJECT> -> [3] -> ['<SUBJECT>']
<ABSTRACT> -> [4] -> ['<ABSTRACT>']
<END> -> [5] -> ['<END>']

All structural tokens are atomic.


Define Dataset Tokenization Function

In [27]:
def tokenize_batch(batch):
    """
    Convert structured text records into
    BPE token IDs.

    The attention mask contains 1 for
    every real token.
    """

    encodings = tokenizer.encode_batch(
        batch["text"]
    )

    input_ids = [
        encoding.ids
        for encoding in encodings
    ]

    attention_masks = [
        [1] * len(encoding.ids)
        for encoding in encodings
    ]

    token_counts = [
        len(encoding.ids)
        for encoding in encodings
    ]

    return {
        "input_ids": input_ids,
        "attention_mask": (
            attention_masks
        ),
        "token_count": token_counts
    }

Test the Tokenization Function

In [28]:
test_batch = {
    "text": [
        structured_dataset[
            "train"
        ][0]["text"],
        structured_dataset[
            "train"
        ][1]["text"]
    ]
}

test_batch_output = tokenize_batch(
    test_batch
)

print(
    "First tokenized record length:",
    test_batch_output[
        "token_count"
    ][0]
)

print(
    "Second tokenized record length:",
    test_batch_output[
        "token_count"
    ][1]
)

print(
    "\nFirst 30 token IDs:"
)

print(
    test_batch_output[
        "input_ids"
    ][0][:30]
)

First tokenized record length: 134
Second tokenized record length: 265

First 30 token IDs:
[2, 4404, 580, 4460, 324, 3660, 18, 49, 3436, 1140, 420, 2557, 226, 3, 710, 298, 580, 226, 4, 402, 921, 1788, 18, 1631, 7602, 1242, 17, 522, 2146, 304]


Tokenize the Complete Dataset

In [29]:
tokenized_dataset = (
    structured_dataset.map(
        tokenize_batch,
        batched=True,
        batch_size=1000,
        desc="Tokenizing structured dataset"
    )
)

print(tokenized_dataset)

Tokenizing structured dataset:   0%|          | 0/178549 [00:00<?, ? examples/s]

Tokenizing structured dataset:   0%|          | 0/9919 [00:00<?, ? examples/s]

Tokenizing structured dataset:   0%|          | 0/9920 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count', 'input_ids', 'attention_mask', 'token_count'],
        num_rows: 178549
    })
    validation: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count', 'input_ids', 'attention_mask', 'token_count'],
        num_rows: 9919
    })
    test: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count', 'input_ids', 'attention_mask', 'token_count'],
        num_rows: 9920
    })
})


Inspect a Tokenized Record

In [30]:
tokenized_example = (
    tokenized_dataset[
        "train"
    ][0]
)

print(
    "Token count:",
    tokenized_example[
        "token_count"
    ]
)

print(
    "\nFirst 50 token IDs:"
)

print(
    tokenized_example[
        "input_ids"
    ][:50]
)

print(
    "\nDecoded record:"
)

print(
    tokenizer.decode(
        tokenized_example[
            "input_ids"
        ],
        skip_special_tokens=False
    )[:1500]
)

Token count: 134

First 50 token IDs:
[2, 4404, 580, 4460, 324, 3660, 18, 49, 3436, 1140, 420, 2557, 226, 3, 710, 298, 580, 226, 4, 402, 921, 1788, 18, 1631, 7602, 1242, 17, 522, 2146, 304, 2519, 2510, 437, 291, 1022, 642, 304, 1719, 279, 3097, 2753, 356, 271, 7602, 14, 296, 264, 1434, 2368, 359]

Decoded record:
<TITLE> Contrastive Language Adaptation for Cross-Lingual Stance Detection <SUBJECT> Computation and Language <ABSTRACT> We study cross-lingual stance detection, which aims to leverage labeled data in one language to identify the relative perspective (or stance) of a given document with respect to a claim in a different target language. In particular, we introduce a novel contrastive language adaptation approach applied to memory networks, which ensures accurate alignment of stances in the source and target languages, and can effectively deal with the challenge of limited labeled data in the target language. The evaluation results on public benchmark datasets and comparison ag

Validate Token ID Range

In [31]:
def validate_token_range(
    dataset_split,
    split_name,
    sample_limit=5000
):

    number_to_check = min(
        sample_limit,
        len(dataset_split)
    )

    minimum_token_id = (
        actual_vocab_size
    )

    maximum_token_id = -1

    for index in range(
        number_to_check
    ):

        token_ids = (
            dataset_split[
                index
            ]["input_ids"]
        )

        if not token_ids:
            continue

        minimum_token_id = min(
            minimum_token_id,
            min(token_ids)
        )

        maximum_token_id = max(
            maximum_token_id,
            max(token_ids)
        )

    is_valid = (
        minimum_token_id >= 0
        and maximum_token_id
        < actual_vocab_size
    )

    print(
        f"{split_name}: "
        f"minimum ID = {minimum_token_id}, "
        f"maximum ID = {maximum_token_id}, "
        f"valid = {is_valid}"
    )

    return is_valid

In [32]:
token_range_checks = {}

for split_name in [
    "train",
    "validation",
    "test"
]:

    token_range_checks[
        split_name
    ] = validate_token_range(
        tokenized_dataset[
            split_name
        ],
        split_name
    )

train: minimum ID = 2, maximum ID = 7999, valid = True
validation: minimum ID = 2, maximum ID = 7999, valid = True
test: minimum ID = 2, maximum ID = 7999, valid = True


Verify Every Record Ends with <END>

In [33]:
END_TOKEN_ID = (
    special_token_ids["<END>"]
)

def check_end_tokens(
    dataset_split,
    sample_limit=5000
):

    number_to_check = min(
        sample_limit,
        len(dataset_split)
    )

    valid_end_count = 0
    empty_count = 0

    for index in range(
        number_to_check
    ):

        token_ids = (
            dataset_split[
                index
            ]["input_ids"]
        )

        if not token_ids:

            empty_count += 1
            continue

        if token_ids[-1] == END_TOKEN_ID:

            valid_end_count += 1

    return {
        "checked": number_to_check,
        "valid_end": valid_end_count,
        "empty": empty_count
    }

In [34]:
for split_name in [
    "train",
    "validation",
    "test"
]:

    end_result = check_end_tokens(
        tokenized_dataset[
            split_name
        ]
    )

    print(
        f"{split_name.capitalize()}: "
        f"{end_result['valid_end']:,}/"
        f"{end_result['checked']:,} "
        "records end with <END>"
    )

    print(
        "Empty records:",
        end_result["empty"]
    )

Train: 5,000/5,000 records end with <END>
Empty records: 0
Validation: 5,000/5,000 records end with <END>
Empty records: 0
Test: 5,000/5,000 records end with <END>
Empty records: 0


Calculate Token Length Statistics

In [35]:
token_length_statistics = {}

for split_name in [
    "train",
    "validation",
    "test"
]:

    token_counts = np.array(
        tokenized_dataset[
            split_name
        ]["token_count"],
        dtype=np.int64
    )

    split_statistics = {
        "records": int(
            len(token_counts)
        ),
        "total_tokens": int(
            token_counts.sum()
        ),
        "minimum_tokens": int(
            token_counts.min()
        ),
        "maximum_tokens": int(
            token_counts.max()
        ),
        "mean_tokens": float(
            token_counts.mean()
        ),
        "median_tokens": float(
            np.median(token_counts)
        ),
        "percentile_90": float(
            np.percentile(
                token_counts,
                90
            )
        ),
        "percentile_95": float(
            np.percentile(
                token_counts,
                95
            )
        ),
        "percentile_99": float(
            np.percentile(
                token_counts,
                99
            )
        )
    }

    token_length_statistics[
        split_name
    ] = split_statistics

    print(
        "=" * 70
    )

    print(
        split_name.upper()
    )

    for key, value in (
        split_statistics.items()
    ):

        if isinstance(value, float):

            print(
                f"{key}: {value:.2f}"
            )

        else:

            print(
                f"{key}: {value:,}"
            )

TRAIN
records: 178,549
total_tokens: 45,356,215
minimum_tokens: 59
maximum_tokens: 1,243
mean_tokens: 254.03
median_tokens: 249.00
percentile_90: 351.00
percentile_95: 382.00
percentile_99: 438.00
VALIDATION
records: 9,919
total_tokens: 2,515,850
minimum_tokens: 62
maximum_tokens: 650
mean_tokens: 253.64
median_tokens: 249.00
percentile_90: 350.00
percentile_95: 382.00
percentile_99: 437.82
TEST
records: 9,920
total_tokens: 2,514,500
minimum_tokens: 70
maximum_tokens: 770
mean_tokens: 253.48
median_tokens: 248.00
percentile_90: 353.00
percentile_95: 384.00
percentile_99: 438.00


Display Token-Length Summary Table

In [36]:
statistics_rows = []

for split_name, values in (
    token_length_statistics.items()
):

    statistics_rows.append({
        "Split": split_name,
        "Records": values[
            "records"
        ],
        "Total Tokens": values[
            "total_tokens"
        ],
        "Mean Tokens": round(
            values["mean_tokens"],
            2
        ),
        "Median Tokens": round(
            values["median_tokens"],
            2
        ),
        "90th Percentile": round(
            values["percentile_90"],
            2
        ),
        "95th Percentile": round(
            values["percentile_95"],
            2
        ),
        "Maximum Tokens": values[
            "maximum_tokens"
        ]
    })

token_statistics_dataframe = (
    pd.DataFrame(
        statistics_rows
    )
)

token_statistics_dataframe

,Split,Records,Total Tokens,Mean Tokens,Median Tokens,90th Percentile,95th Percentile,Maximum Tokens
0,train,178549,45356215,254.03,249.0,351.0,382.0,1243
1,validation,9919,2515850,253.64,249.0,350.0,382.0,650
2,test,9920,2514500,253.48,248.0,353.0,384.0,770


Analyze Coverage for Different Context Lengths

In [37]:
CONTEXT_LENGTHS = [
    128,
    256,
    512,
    1024
]

training_token_counts = np.array(
    tokenized_dataset[
        "train"
    ]["token_count"],
    dtype=np.int64
)

for context_length in (
    CONTEXT_LENGTHS
):

    fitting_records = np.sum(
        training_token_counts
        <= context_length
    )

    coverage_percentage = (
        fitting_records /
        len(training_token_counts)
        * 100
    )

    print(
        f"Context length {context_length}: "
        f"{coverage_percentage:.2f}% "
        "of records fit completely"
    )

Context length 128: 2.93% of records fit completely
Context length 256: 54.11% of records fit completely
Context length 512: 99.86% of records fit completely
Context length 1024: 100.00% of records fit completely


Remove Existing Tokenized Dataset

In [38]:
if os.path.exists(
    TOKENIZED_DATASET_PATH
):

    print(
        "Removing existing tokenized dataset:"
    )

    print(
        TOKENIZED_DATASET_PATH
    )

    shutil.rmtree(
        TOKENIZED_DATASET_PATH
    )

Save Tokenized Hugging Face Dataset

In [39]:
tokenized_dataset.save_to_disk(
    TOKENIZED_DATASET_PATH
)

print(
    "Tokenized dataset saved successfully."
)

print(
    "Location:",
    TOKENIZED_DATASET_PATH
)

Saving the dataset (0/2 shards):   0%|          | 0/178549 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9919 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9920 [00:00<?, ? examples/s]

Tokenized dataset saved successfully.
Location: /content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenized_bpe_dataset


Define Memory-Efficient Token Stream Export

Instead of first creating one very large Python list, this function writes token IDs directly into a NumPy memory-mapped file.

This reduces RAM usage.

In [40]:
def create_token_stream(
    dataset_split,
    output_path
):
    """
    Flatten document token IDs into one
    compact uint16 NumPy token stream.

    uint16 is safe because the vocabulary
    size is below 65,536.
    """

    if actual_vocab_size >= 65536:

        raise ValueError(
            "Vocabulary is too large for "
            "uint16 token storage."
        )

    total_tokens = int(
        sum(
            dataset_split[
                "token_count"
            ]
        )
    )

    token_stream = np.memmap(
        output_path,
        dtype=np.uint16,
        mode="w+",
        shape=(total_tokens,)
    )

    write_position = 0

    for index, example in enumerate(
        dataset_split
    ):

        token_ids = np.asarray(
            example["input_ids"],
            dtype=np.uint16
        )

        next_position = (
            write_position +
            len(token_ids)
        )

        token_stream[
            write_position:
            next_position
        ] = token_ids

        write_position = (
            next_position
        )

        if (
            index > 0
            and index % 10000 == 0
        ):

            print(
                f"Written {index:,} records"
            )

    token_stream.flush()

    del token_stream

    return total_tokens

In [41]:
TRAIN_STREAM_PATH = os.path.join(
    TOKEN_STREAM_PATH,
    "train_tokens.bin"
)

VALIDATION_STREAM_PATH = os.path.join(
    TOKEN_STREAM_PATH,
    "validation_tokens.bin"
)

TEST_STREAM_PATH = os.path.join(
    TOKEN_STREAM_PATH,
    "test_tokens.bin"
)

TOKEN_STREAM_METADATA_PATH = os.path.join(
    TOKEN_STREAM_PATH,
    "stream_metadata.json"
)

print("Training stream:", TRAIN_STREAM_PATH)
print("Validation stream:", VALIDATION_STREAM_PATH)
print("Test stream:", TEST_STREAM_PATH)

Training stream: /content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenized_bpe_streams/train_tokens.bin
Validation stream: /content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenized_bpe_streams/validation_tokens.bin
Test stream: /content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenized_bpe_streams/test_tokens.bin


In [42]:
train_total_tokens = (
    create_token_stream(
        tokenized_dataset[
            "train"
        ],
        TRAIN_STREAM_PATH
    )
)

print(
    "Training tokens exported:",
    f"{train_total_tokens:,}"
)

Written 10,000 records
Written 20,000 records
Written 30,000 records
Written 40,000 records
Written 50,000 records
Written 60,000 records
Written 70,000 records
Written 80,000 records
Written 90,000 records
Written 100,000 records
Written 110,000 records
Written 120,000 records
Written 130,000 records
Written 140,000 records
Written 150,000 records
Written 160,000 records
Written 170,000 records
Training tokens exported: 45,356,215


In [43]:
validation_total_tokens = (
    create_token_stream(
        tokenized_dataset[
            "validation"
        ],
        VALIDATION_STREAM_PATH
    )
)

print(
    "Validation tokens exported:",
    f"{validation_total_tokens:,}"
)

Validation tokens exported: 2,515,850


In [44]:
test_total_tokens = (
    create_token_stream(
        tokenized_dataset[
            "test"
        ],
        TEST_STREAM_PATH
    )
)

print(
    "Test tokens exported:",
    f"{test_total_tokens:,}"
)

Test tokens exported: 2,514,500


Save Token Stream Metadata

In [45]:
stream_metadata = {
    "dtype": "uint16",
    "vocab_size": (
        actual_vocab_size
    ),
    "special_token_ids": (
        special_token_ids
    ),
    "streams": {
        "train": {
            "path": (
                TRAIN_STREAM_PATH
            ),
            "number_of_tokens": (
                train_total_tokens
            )
        },
        "validation": {
            "path": (
                VALIDATION_STREAM_PATH
            ),
            "number_of_tokens": (
                validation_total_tokens
            )
        },
        "test": {
            "path": (
                TEST_STREAM_PATH
            ),
            "number_of_tokens": (
                test_total_tokens
            )
        }
    }
}

with open(
    TOKEN_STREAM_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        stream_metadata,
        file,
        indent=2,
        ensure_ascii=False
    )

print(
    "Token-stream metadata saved:"
)

print(
    TOKEN_STREAM_METADATA_PATH
)

Token-stream metadata saved:
/content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenized_bpe_streams/stream_metadata.json


In [46]:
stream_files = {
    "train": TRAIN_STREAM_PATH,
    "validation": (
        VALIDATION_STREAM_PATH
    ),
    "test": TEST_STREAM_PATH
}

expected_token_counts = {
    "train": train_total_tokens,
    "validation": (
        validation_total_tokens
    ),
    "test": test_total_tokens
}

for split_name, file_path in (
    stream_files.items()
):

    file_size_bytes = os.path.getsize(
        file_path
    )

    expected_size_bytes = (
        expected_token_counts[
            split_name
        ] *
        np.dtype(
            np.uint16
        ).itemsize
    )

    print(
        f"{split_name.capitalize()}:"
    )

    print(
        "File exists:",
        os.path.exists(file_path)
    )

    print(
        "Actual size:",
        f"{file_size_bytes / (1024 ** 2):.2f} MB"
    )

    print(
        "Size is correct:",
        file_size_bytes
        == expected_size_bytes
    )

    print("-" * 60)

Train:
File exists: True
Actual size: 86.51 MB
Size is correct: True
------------------------------------------------------------
Validation:
File exists: True
Actual size: 4.80 MB
Size is correct: True
------------------------------------------------------------
Test:
File exists: True
Actual size: 4.80 MB
Size is correct: True
------------------------------------------------------------


Load a Token Stream with Memory Mapping

This demonstrates how Notebook 4 will load the data without placing the complete token stream into RAM.

In [47]:
train_token_stream = np.memmap(
    TRAIN_STREAM_PATH,
    dtype=np.uint16,
    mode="r"
)

validation_token_stream = np.memmap(
    VALIDATION_STREAM_PATH,
    dtype=np.uint16,
    mode="r"
)

test_token_stream = np.memmap(
    TEST_STREAM_PATH,
    dtype=np.uint16,
    mode="r"
)

print(
    "Training stream shape:",
    train_token_stream.shape
)

print(
    "Validation stream shape:",
    validation_token_stream.shape
)

print(
    "Test stream shape:",
    test_token_stream.shape
)

Training stream shape: (45356215,)
Validation stream shape: (2515850,)
Test stream shape: (2514500,)


Decode a Portion of the Training Stream

In [48]:
stream_preview_ids = (
    train_token_stream[
        :300
    ].astype(
        np.int64
    ).tolist()
)

stream_preview_text = (
    tokenizer.decode(
        stream_preview_ids,
        skip_special_tokens=False
    )
)

print(stream_preview_text)

<TITLE> Contrastive Language Adaptation for Cross-Lingual Stance Detection <SUBJECT> Computation and Language <ABSTRACT> We study cross-lingual stance detection, which aims to leverage labeled data in one language to identify the relative perspective (or stance) of a given document with respect to a claim in a different target language. In particular, we introduce a novel contrastive language adaptation approach applied to memory networks, which ensures accurate alignment of stances in the source and target languages, and can effectively deal with the challenge of limited labeled data in the target language. The evaluation results on public benchmark datasets and comparison against current state-of-the-art approaches demonstrate the effectiveness of our approach. <END><TITLE> Do Neural Networks Need Gradient Descent to Generalize? A Theoretical Study <SUBJECT> Machine Learning <ABSTRACT> Conventional wisdom attributes the mysterious generalization abilities of overparameterized neural 

Test a Training Batch Window

The improved GPT will use a context length of 256 tokens.

In [49]:
BLOCK_SIZE = 256
BATCH_SIZE = 4

maximum_start = (
    len(train_token_stream)
    - BLOCK_SIZE
    - 1
)

random_starts = np.random.randint(
    low=0,
    high=maximum_start,
    size=BATCH_SIZE
)

input_batch = np.stack([
    np.asarray(
        train_token_stream[
            start:
            start + BLOCK_SIZE
        ],
        dtype=np.int64
    )
    for start in random_starts
])

target_batch = np.stack([
    np.asarray(
        train_token_stream[
            start + 1:
            start + BLOCK_SIZE + 1
        ],
        dtype=np.int64
    )
    for start in random_starts
])

input_batch = torch.tensor(
    input_batch,
    dtype=torch.long
)

target_batch = torch.tensor(
    target_batch,
    dtype=torch.long
)

print(
    "Input batch shape:",
    input_batch.shape
)

print(
    "Target batch shape:",
    target_batch.shape
)

print(
    "\nFirst input sequence preview:"
)

print(
    tokenizer.decode(
        input_batch[
            0
        ].tolist(),
        skip_special_tokens=False
    )[:800]
)

Input batch shape: torch.Size([4, 256])
Target batch shape: torch.Size([4, 256])

First input sequence preview:
 sequences are affected by the need for more information in memory. This paper introduces Long Term Memory network (LTM), which can tackle the exploding and vanishing gradient problems and handles long sequences without forgetting. LTM is designed to scale data in the memory and gives a higher weight to the input in the sequence. LTM avoid overfitting by scaling the cell state after achieving the optimal results. The LTM is tested on Penn treebank dataset, and Text8 dataset and LTM achieves test perplexities of 83 and 82 respectively. 650 LTM cells achieved a test perplexity of 67 for Penn treebank, and 600 cells achieved a test perplexity of 77 for Text8. LTM achieves state of the art results by only using ten hidden LTM cells for both datasets. <END><TITLE> FERMAT: An Alternative to Accu


In [50]:
tokenizer_configuration = {
    "tokenizer_type": (
        "ByteLevelBPE"
    ),
    "vocab_size_target": (
        VOCAB_SIZE
    ),
    "vocab_size_actual": (
        actual_vocab_size
    ),
    "minimum_frequency": (
        MIN_FREQUENCY
    ),
    "special_tokens": (
        SPECIAL_TOKENS
    ),
    "special_token_ids": (
        special_token_ids
    ),
    "pad_token": "<PAD>",
    "pad_token_id": (
        special_token_ids[
            "<PAD>"
        ]
    ),
    "unk_token": "<UNK>",
    "unk_token_id": (
        special_token_ids[
            "<UNK>"
        ]
    ),
    "title_token": "<TITLE>",
    "title_token_id": (
        special_token_ids[
            "<TITLE>"
        ]
    ),
    "subject_token": (
        "<SUBJECT>"
    ),
    "subject_token_id": (
        special_token_ids[
            "<SUBJECT>"
        ]
    ),
    "abstract_token": (
        "<ABSTRACT>"
    ),
    "abstract_token_id": (
        special_token_ids[
            "<ABSTRACT>"
        ]
    ),
    "end_token": "<END>",
    "end_token_id": (
        special_token_ids[
            "<END>"
        ]
    ),
    "recommended_block_size": (
        BLOCK_SIZE
    ),
    "tokenizer_json_path": (
        TOKENIZER_JSON_PATH
    ),
    "tokenized_dataset_path": (
        TOKENIZED_DATASET_PATH
    ),
    "token_stream_path": (
        TOKEN_STREAM_PATH
    )
}

with open(
    TOKENIZER_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        tokenizer_configuration,
        file,
        indent=2,
        ensure_ascii=False
    )

print(
    "Tokenizer configuration saved:"
)

print(
    TOKENIZER_CONFIG_PATH
)

Tokenizer configuration saved:
/content/drive/MyDrive/Scientific-Abstract-GPT/data/bpe_tokenizer/tokenizer_config.json


Save Complete Tokenization Summary

In [51]:
tokenization_summary = {
    "dataset": {
        "processed_dataset_path": (
            PROCESSED_DATASET_PATH
        ),
        "records": {
            split_name: len(
                tokenized_dataset[
                    split_name
                ]
            )
            for split_name in [
                "train",
                "validation",
                "test"
            ]
        }
    },
    "tokenizer": (
        tokenizer_configuration
    ),
    "token_statistics": (
        token_length_statistics
    ),
    "token_streams": (
        stream_metadata
    ),
    "validation": {
        "special_tokens_are_valid": (
            len(
                invalid_special_tokens
            ) == 0
        ),
        "token_ranges_are_valid": (
            all(
                token_range_checks.values()
            )
        )
    }
}

with open(
    TOKENIZATION_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        tokenization_summary,
        file,
        indent=2,
        ensure_ascii=False
    )

print(
    "Tokenization summary saved:"
)

print(
    TOKENIZATION_SUMMARY_PATH
)

Tokenization summary saved:
/content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenization_summary.json


In [52]:
reloaded_tokenized_dataset = (
    load_from_disk(
        TOKENIZED_DATASET_PATH
    )
)

print(
    reloaded_tokenized_dataset
)

reloaded_example = (
    reloaded_tokenized_dataset[
        "train"
    ][0]
)

print(
    "\nReloaded token count:",
    reloaded_example[
        "token_count"
    ]
)

print(
    "\nReloaded decoded example:"
)

print(
    tokenizer.decode(
        reloaded_example[
            "input_ids"
        ],
        skip_special_tokens=False
    )[:1200]
)

DatasetDict({
    train: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count', 'input_ids', 'attention_mask', 'token_count'],
        num_rows: 178549
    })
    validation: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count', 'input_ids', 'attention_mask', 'token_count'],
        num_rows: 9919
    })
    test: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count', 'input_ids', 'attention_mask', 'token_count'],
        num_rows: 9920
    })
})

Reloaded token count: 134

Reloaded decoded example:
<TITLE> Contrastive Language Adaptation for Cross-Lingual Stance Detection <SUBJECT> Computation and Language <ABSTRACT> We study cross-lingual stance detection, which aims to leverage labeled data in one language to identif

Final Validation Checks

In [53]:
validation_checks = {
    "tokenizer_json_saved": (
        os.path.exists(
            TOKENIZER_JSON_PATH
        )
    ),
    "tokenizer_config_saved": (
        os.path.exists(
            TOKENIZER_CONFIG_PATH
        )
    ),
    "tokenized_dataset_saved": (
        os.path.exists(
            TOKENIZED_DATASET_PATH
        )
    ),
    "training_stream_saved": (
        os.path.exists(
            TRAIN_STREAM_PATH
        )
    ),
    "validation_stream_saved": (
        os.path.exists(
            VALIDATION_STREAM_PATH
        )
    ),
    "test_stream_saved": (
        os.path.exists(
            TEST_STREAM_PATH
        )
    ),
    "stream_metadata_saved": (
        os.path.exists(
            TOKEN_STREAM_METADATA_PATH
        )
    ),
    "tokenization_summary_saved": (
        os.path.exists(
            TOKENIZATION_SUMMARY_PATH
        )
    ),
    "vocabulary_is_not_empty": (
        actual_vocab_size > 0
    ),
    "special_tokens_valid": (
        len(
            invalid_special_tokens
        ) == 0
    ),
    "token_ranges_valid": (
        all(
            token_range_checks.values()
        )
    ),
    "training_stream_not_empty": (
        train_total_tokens > 0
    ),
    "validation_stream_not_empty": (
        validation_total_tokens > 0
    ),
    "test_stream_not_empty": (
        test_total_tokens > 0
    )
}

for check_name, result in (
    validation_checks.items()
):

    status = (
        "PASSED"
        if result
        else "FAILED"
    )

    print(
        f"{check_name}: {status}"
    )

tokenizer_json_saved: PASSED
tokenizer_config_saved: PASSED
tokenized_dataset_saved: PASSED
training_stream_saved: PASSED
validation_stream_saved: PASSED
test_stream_saved: PASSED
stream_metadata_saved: PASSED
tokenization_summary_saved: PASSED
vocabulary_is_not_empty: PASSED
special_tokens_valid: PASSED
token_ranges_valid: PASSED
training_stream_not_empty: PASSED
validation_stream_not_empty: PASSED
test_stream_not_empty: PASSED


Completion Message

In [55]:
if all(
    validation_checks.values()
):

    print("=" * 80)

    print(
        "02_Tokenization.ipynb "
        "completed successfully."
    )

    print("=" * 80)

    print(
        "\nTokenizer type:"
    )

    print(
        "Byte-Level Byte Pair Encoding"
    )

    print(
        "\nVocabulary size:"
    )

    print(
        f"{actual_vocab_size:,}"
    )

    print(
        "\nStructural tokens:"
    )

    for token, token_id in (
        special_token_ids.items()
    ):

        print(
            f"{token}: {token_id}"
        )

    print(
        "\nTraining tokens:"
    )

    print(
        f"{train_total_tokens:,}"
    )

    print(
        "\nThe training notebook should "
        "load the memory-mapped token "
        "streams from:"
    )

    print(
        TOKEN_STREAM_PATH
    )

else:

    failed_checks = [
        check_name
        for check_name, result
        in validation_checks.items()
        if not result
    ]

    raise RuntimeError(
        "Tokenization validation failed: "
        f"{failed_checks}"
    )

02_Tokenization.ipynb completed successfully.

Tokenizer type:
Byte-Level Byte Pair Encoding

Vocabulary size:
8,000

Structural tokens:
<PAD>: 0
<UNK>: 1
<TITLE>: 2
<SUBJECT>: 3
<ABSTRACT>: 4
<END>: 5

Training tokens:
45,356,215

The training notebook should load the memory-mapped token streams from:
/content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenized_bpe_streams
